# Démo — Assistant RAG + Forecast pour KPI Logistiques

Ce notebook illustre l'architecture du projet de bout en bout :
1. Génération / chargement des données synthétiques
2. Construction du pipeline RAG (chunking -> embeddings -> index FAISS)
3. Module de forecast sur les KPI numériques
4. Agent léger qui route chaque question vers le bon outil (RAG, forecast, ou les deux)

> ⚠️ Toutes les données utilisées ici sont **100% synthétiques**, générées par `data/generate_synthetic_data.py`. Aucune donnée réelle n'est utilisée.

## 0. Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import pandas as pd
from src import config
from src.rag_pipeline import RagPipeline
from src.forecast import ForecastEngine
from src.agent import LogisticsAgent

## 1. Aperçu des données synthétiques

(Si les fichiers n'existent pas encore : `!python ../data/generate_synthetic_data.py`)

In [ ]:
df = pd.read_csv(config.KPI_CSV_PATH)
df.head()

In [ ]:
df.set_index("date")["cout_par_tonne_usd"].plot(title="Coût par tonne (USD) — historique simulé", figsize=(10, 3));

## 2. Pipeline RAG — indexation puis interrogation

Chunking -> embeddings TF-IDF (par défaut, sans dépendance externe) -> index FAISS -> retrieval -> synthèse de la réponse.

In [ ]:
rag = RagPipeline()
rag.build_index()

In [ ]:
result = rag.answer("Que dit la procédure de gestion des incidents HSE en cas d'incident majeur ?")
print(result["answer"])
print("\nSources:", result["sources"])

## 3. Module de forecast

Lissage exponentiel Holt-Winters (tendance + saisonnalité), avec repli automatique sur une régression linéaire si la série ne s'y prête pas.

In [ ]:
fe = ForecastEngine()
forecast = fe.forecast("cout_par_tonne_usd", horizon_months=3)
print(forecast.method)
print(forecast.explanation)
list(zip(forecast.forecast_dates, forecast.forecast_values))

## 4. Agent — routage automatique RAG vs Forecast ("IA agentique")

L'agent décide, question par question, quel(s) outil(s) appeler : recherche documentaire, prévision numérique, ou les deux pour une question hybride.

In [ ]:
agent = LogisticsAgent()

questions = [
    "Quelle sera l'évolution du coût par tonne dans les 3 prochains mois ?",
    "Pourquoi le taux de disponibilité de la flotte a-t-il baissé en novembre 2023 ?",
    "Quels incidents HSE ont eu lieu récemment et comment va évoluer le délai de livraison dans les 2 prochains mois ?",
]

for q in questions:
    r = agent.ask(q)
    print(f"Q: {q}")
    print(f"  Outils utilisés: {r.tools_used} (routage: {r.routing_method})")
    print(f"  Réponse (extrait): {r.answer[:220]}...")
    print()

## 5. Limites et pistes d'amélioration

- **Données synthétiques** : le projet est un démonstrateur d'architecture, pas un modèle validé sur des données réelles.
- **Génération extractive par défaut** : sans clé API, les réponses RAG sont composées par extraction/agrégation des passages pertinents plutôt que reformulées par un LLM. Un backend OpenAI (ou un modèle local) peut être branché via `src/llm_backend.py`.
- **Embeddings TF-IDF** : suffisants pour un petit corpus de démonstration ; un modèle d'embedding dense (sentence-transformers, embeddings OpenAI) améliorerait la qualité de retrieval sur un corpus plus large et plus hétérogène — l'architecture (`src/embeddings.py`) est conçue pour permettre ce remplacement sans changer le reste du pipeline.
- **Routage agentique** : le mode par défaut est à base de règles (déterministe, sans coût API) ; un mode `openai_function_calling` est disponible si une clé API est fournie.